# IMEN266 · HW 2 workspace (Ch.4)

Statement: `hw2.pdf` (PLMS / repo). This notebook is your **computational
workspace** for Parts B–C and your **prompt log** for Part D.
Part A is pen-and-paper first — no cells here on purpose.

▶ Colab: `https://colab.research.google.com/github/youngmko/imen266-2026/blob/main/ch04/homework/HW2.ipynb`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
try:
    from imen266.dtmc import DTMC
except ImportError:                                  # Colab: fetch the course package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/youngmko/imen266-2026.git"], check=True)
    from imen266.dtmc import DTMC
rng = np.random.default_rng()      # your results should NOT depend on a seed
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": .3})

---
## Part B2 — Correct transition matrix and numbers (formulas on paper, numbers here)

Fill `P_inv` with the exact Poisson(3) expressions you derived (`stats.poisson.pmf`
and `stats.poisson.sf`), then solve for $\pi$ and compute the two requested
quantities.

In [ ]:
# --- B2 skeleton -------------------------------------------------------------
S_inv = [2, 3, 4, 5]
lam = 3.0
P_inv = np.full((4, 4), np.nan)        # TODO: fill every entry (rows must sum to 1 without normalising!)

if np.isnan(P_inv).any():
    print("fill P_inv first")
else:
    himart = DTMC(P_inv, S_inv)
    pi = himart.stationary()
    p_order_given_state = np.array([stats.poisson.sf(i - 2, lam) for i in S_inv])   # P(D >= i-1)
    p_order = float(pi @ p_order_given_state)
    a = np.array([0, 0, 0, 1.0])
    p_x4_eq_3 = himart.distribution(a, 4)[S_inv.index(3)]
    print("pi =", pi.round(4)); print(f"P(order) long run = {p_order:.4f}   P(X_4 = 3 | X_0 = 5) = {p_x4_eq_3:.4f}")

---
## Part B3 — Simulate the store policy directly (no matrix!)

In [ ]:
# --- B3 skeleton -------------------------------------------------------------
weeks = 200_000
monday = np.empty(weeks + 1, int); order = np.zeros(weeks, bool)
monday[0] = 5
for n in range(weeks):
    D = rng.poisson(3)
    friday = max(monday[n] - D, 0)               # lost sales
    order[n] = friday < 2
    monday[n + 1] = 5 if order[n] else friday

p_hat = order.mean()
se = np.sqrt(p_hat * (1 - p_hat) / weeks)        # (ignores autocorrelation: a slightly optimistic CI)
print(f"long-run P(order) ≈ {p_hat:.4f}   95% CI ± {1.96*se:.4f}")

# TODO: P(X_4 = 3 | X_0 = 5): run R independent 4-week histories starting from 5 and count
R = 100_000
# ... your code ...
# print(f"P(X_4 = 3) ≈ {...:.4f}   95% CI ± {...:.4f}")

---
## Part C2 — Fit a 2-state chain to weather that has memory

Truth: spell model (CP3 #7) with $p_k=(0.6,0.7,0.8,0.9)$, $q_k=(0.5,0.4,0.3,0.2)$.
Observation: only S/R per day. Fit: $\hat P_{ij}=\#\{i\to j\}/\#\{i\}$.

In [ ]:
# --- C2 skeleton -------------------------------------------------------------
p_k = np.array([0.6, 0.7, 0.8, 0.9]); q_k = np.array([0.5, 0.4, 0.3, 0.2])
N = 100_000

# simulate the TRUTH: (type, day-of-spell) -> record only the type (1 = sunny, 0 = rainy)
x = np.empty(N, int); typ, k = 1, 1
for n in range(N):
    x[n] = typ
    stay = p_k[min(k, 4) - 1] if typ == 1 else q_k[min(k, 4) - 1]
    if rng.random() < stay: k += 1
    else: typ, k = 1 - typ, 1

# fit a 2-state chain by transition frequencies (states: 0 = R, 1 = S)
Nt = np.zeros((2, 2)); np.add.at(Nt, (x[:-1], x[1:]), 1)
P_hat = Nt / Nt.sum(axis=1, keepdims=True)
fit = DTMC(P_hat, ["R", "S"])
print("fitted P_hat =\n", P_hat.round(4))

# (a) long-run fraction of sunny days: truth (empirical) vs fit (pi)
print(f"(a) sunny fraction: truth {x.mean():.4f}   fit {fit.stationary()[1]:.4f}")

# (b) mean sunny-spell length: truth (empirical) vs fit (geometric: 1/(1-P_hat[S,S]))
edges = np.diff(np.r_[0, x, 0]); L = np.where(edges == -1)[0] - np.where(edges == 1)[0]
print(f"(b) mean sunny spell: truth {L.mean():.3f}   fit {1/(1-P_hat[1,1]):.3f}")

# TODO (c): P(sunny in 3 days | sunny today): truth = empirical from x; fit = (P_hat^3)[S,S]
# TODO (d): distribution of spell lengths: histogram of L vs geometric pmf (1-p)p^(k-1) with p = P_hat[S,S]
# TODO (e): P(sunny tomorrow | sunny today AND yesterday): truth = empirical; fit = P_hat[S,S] (why?)

**C2(f) — chi-square test of the Markov property** (the December programming
assignment uses the same test). $H_0$: $P(X_{n+1}\mid X_n=k, X_{n-1}=i)$ does
not depend on $i$. For each current state $k$ build the $2\times2$ table of
$(X_{n-1}, X_{n+1})$ counts and test independence.

In [ ]:
# --- C2(f) skeleton ----------------------------------------------------------
def markov_test(seq):
    # Chi-square test of the Markov property: one 2x2 table (prev x next) per current state k.
    # Returns {k: (chi2, p_value), "combined_p": Fisher-combined p-value}.
    T = np.zeros((2, 2, 2), int)                                   # T[prev, curr, next]
    np.add.at(T, (seq[:-2], seq[1:-1], seq[2:]), 1)
    out = {}
    for k in range(2):
        chi2, p, dof, _ = stats.chi2_contingency(T[:, k, :])
        out[k] = (chi2, p)
    fisher = -2 * sum(np.log(max(p, 1e-300)) for _, p in out.values())
    out["combined_p"] = stats.chi2.sf(fisher, 2 * 2)
    return out

res = markov_test(x)
print("spell model (truth has memory): p-values per current state:",
      {k: f"{v[1]:.1e}" for k, v in res.items() if k != "combined_p"}, " combined:", f"{res['combined_p']:.1e}")
# TODO: simulate 100,000 days from the FITTED chain -- fit.sample_path(100_000, x0="S", rng=rng) -- and run
#       markov_test on it. What p-values do you expect when H0 is true? (Repeat a few times.)

**C2 write-up (≤5 sentences):** *(double-click to edit)*

> ...

**C1 (before running!):** predictions (a)–(e) with one-line reasons: ___

---
## Part C3 (optional) — How stable is the PageRank top-10 in $\beta$?

In [ ]:
# --- C3 skeleton -------------------------------------------------------------
from imen266.pagerank import link_matrix, pagerank
n = 200
edges = [(u, int(v)) for u in range(n) for v in rng.choice(n, size=rng.integers(1, 6), replace=False) if v != u]
P = link_matrix(edges, n)
top_ref = set(np.argsort(pagerank(P, beta=0.85))[-10:])
betas = np.linspace(0.5, 0.99, 25)
# TODO: for each beta compute the top-10 set and its overlap with top_ref; plot overlap vs beta.

---
## Part D — Prompt log (mandatory if AI was used)

| # | Where I used AI | Prompt (verbatim or faithful summary) | What it returned | What I verified / corrected |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |

**Reflection (2–3 sentences):** where was the AI most useful, and where was it
least trustworthy, on this homework?

> ...